# Phase 0 spike — v2

**v1 ran on 2026-09-21 (vLLM 0.29.0, Qwen3-0.6B, Kaggle 2x T4) and the gate PASSED.** Its results are
in `artifacts/results/phase0_kaggle_t4/`. The prefix-cache counters exist, move, count **tokens**, and
our cache model now reproduces them to the token.

**v2 exists because v1 got three secondary verdicts wrong — all three were bugs in the notebook, not
findings about vLLM:**

| v1 said | What actually happened | v2 fix |
|---|---|---|
| caching ON does **not** beat OFF | With caching off vLLM never queries the cache, so a hit *rate* is undefined. The real comparison — prefill tokens actually computed — was **64 vs 32,832, a 513x difference** | Compare `vllm:prompt_tokens_by_source{source="local_compute"}` |
| `kv_block_*` histograms absent | The enable flag is **`--kv-cache-metrics`**. `--kv-cache-metrics-sample` only sets the rate (default 0.01). And `vllm serve --help` did not list it, so the probe was a false negative | Pass both flags; verify from the engine's own config dump and the scrape — never from `--help` |
| two servers **cannot** coexist | Both died with *"4.38 GiB KV needed for max seq len 40960, only 2.69 GiB available"* — vLLM refused to start because one **40,960-token** request would not fit. Our prompts are 513 tokens | `--max-model-len 4096` |
| (not checked) | **The box has two physical GPUs.** v1 only knew MIG and memory-split, so it crammed both servers onto GPU 0 | One server per physical GPU — true isolation, better than MIG |

## Same rules as v1

Self-contained: no repo, no dataset, no HF token. Only `vllm` + stdlib. Model **`Qwen/Qwen3-0.6B`**
(ungated). Runtime ≈ 25–35 min. Copy `phase0_v2_results/` back into `artifacts/results/` and commit it,
**including the raw `.txt` scrapes**.

## ⚠️ What a T4 can and cannot tell us

The v1 log confirmed ADR-005's prediction exactly: T4 (SM75) has **no bf16** (vLLM fell back to fp16),
**no FlashAttention-2** (*"FA2 is only supported on devices with compute capability >= 8"*), and runs
attention on **TRITON_ATTN**. So on a T4:

- ✅ **Cache accounting, hit rates, the mechanism, the 2-replica routing test** — hardware-independent,
  fully valid.
- ⛔ **TTFT and cost coefficients** — T4's TRITON_ATTN + fp16 path is not how an A100/H100 serves.
  Timing numbers from this box are **T4 numbers** and must be labelled that way. The Phase-3 cost
  calibration belongs on datacenter hardware.


## 0 — Configuration


In [ ]:
MODEL = "Qwen/Qwen3-0.6B"   # ungated. Qwen/Qwen3-8B for the Phase-3 calibration.
BLOCK_SIZE = 16
MAX_MODEL_LEN = 4096        # v1 died on the 40,960 default. Our prompts are ~513 tokens.
PORT_A, PORT_B = 8000, 8001
N_SHARED_REQUESTS = 64
SHARED_PREFIX_TOKENS = 512  # = 32 full blocks
GPU_MEM_UTIL = 0.85         # per server when each server has its own GPU
GPU_MEM_UTIL_SPLIT = 0.42   # each, only when two servers must share one card
STARTUP_TIMEOUT_S = 900
OUT_DIR = "phase0_v2_results"


## 1 — Environment, and how many physical GPUs


In [ ]:
import json
import os
import pathlib
import platform
import random
import re
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone

pathlib.Path(OUT_DIR).mkdir(exist_ok=True)


def sh(cmd, timeout=60):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True,
                              timeout=timeout, check=False).stdout.strip()
    except Exception as e:  # an unfamiliar box: any binary may be missing
        return f"<failed: {e}>"


ENV = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "cpu_count": os.cpu_count(),
    "gpus": sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version "
               "--format=csv,noheader"),
    "nvidia_smi_L": sh("nvidia-smi -L"),
    "mig_mode": sh("nvidia-smi --query-gpu=mig.mode.current --format=csv,noheader"),
}
for k, v in ENV.items():
    print(f"{k:14s} {v}")

PHYSICAL_GPUS = [ln for ln in ENV["nvidia_smi_L"].splitlines() if ln.startswith("GPU ")]
MIG_UUIDS = [ln.split("UUID:")[1].strip().rstrip(")").strip()
             for ln in ENV["nvidia_smi_L"].splitlines() if "MIG-" in ln and "UUID:" in ln]

if len(PHYSICAL_GPUS) >= 2:
    PLANNED_PATH = "multi-gpu"
elif len(MIG_UUIDS) >= 2:
    PLANNED_PATH = "mig"
else:
    PLANNED_PATH = "memory-split"
print(f"\nphysical GPUs: {len(PHYSICAL_GPUS)}   MIG instances: {len(MIG_UUIDS)}")
print(f"2-replica path: {PLANNED_PATH}")
print({"multi-gpu": "  -> separate physical GPUs: true isolation. Hit rate AND TTFT reportable.",
       "mig": "  -> MIG partitions SMs + memory. Hit rate AND TTFT reportable.",
       "memory-split": "  -> shared SMs. Hit rate reportable, TTFT percentiles NOT (ADR-005)."}[PLANNED_PATH])


## 2 — Install vLLM (skip if present)


In [ ]:
!pip install -q vllm


## 3 — Metric-name audit against this version's own source


In [ ]:
import vllm

VLLM_VERSION = vllm.__version__
print("vllm", VLLM_VERSION)
src = pathlib.Path(vllm.__file__).parent / "v1" / "metrics"
blob = "".join(p.read_text(errors="replace") for p in src.rglob("*.py")) if src.exists() else ""
WANTED = ["prefix_cache_queries", "prefix_cache_hits", "prompt_tokens_by_source",
          "prompt_tokens_cached", "time_to_first_token_seconds", "num_requests_waiting",
          "kv_block_lifetime_seconds", "kv_block_idle_before_evict_seconds",
          "kv_block_reuse_gap_seconds"]
AUDIT = {n: n in blob for n in WANTED}
AUDIT["prefix_cache_hit_rate_EXPECTED_ABSENT"] = "prefix_cache_hit_rate" in blob
for k, v in AUDIT.items():
    print(f"  {'!!' if (k.endswith('ABSENT') and v) else ('OK' if v else '--')}  {k}")


## 4 — Server helpers

The metrics flag is **not probed via `--help`** — v1 showed that probe gives false negatives. Instead
the server is started with `--kv-cache-metrics --kv-cache-metrics-sample 1.0`; if it rejects them
(`unrecognized arguments` in the log) it restarts without, and the answer is read from the engine's own
config dump (`kv_cache_metrics=True`), which is authoritative.


In [ ]:
SERVERS = {}          # port -> {proc, logf, log_path, argv}
METRICS_FLAG_OK = None


def start_server(port, tag, prefix_caching=True, device=None, mem=GPU_MEM_UTIL, kv_metrics=True):
    argv = ["vllm", "serve", MODEL, "--port", str(port),
            "--block-size", str(BLOCK_SIZE),
            "--max-model-len", str(MAX_MODEL_LEN),
            "--gpu-memory-utilization", str(mem)]
    # V1 has prefix caching ON by default; the OFF arm needs the explicit negation.
    argv += ["--enable-prefix-caching"] if prefix_caching else ["--no-enable-prefix-caching"]
    if prefix_caching and kv_metrics:
        argv += ["--kv-cache-metrics", "--kv-cache-metrics-sample", "1.0"]
    env = dict(os.environ)
    if device is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(device)
    # One log per ARM. v1 reused server_<port>.log and the later runs overwrote the earlier ones.
    log_path = pathlib.Path(OUT_DIR) / f"server_{port}_{tag}.log"
    logf = open(log_path, "w")  # noqa: SIM115 - must outlive this cell
    proc = subprocess.Popen(argv, stdout=logf, stderr=subprocess.STDOUT, env=env)
    SERVERS[port] = {"proc": proc, "logf": logf, "log_path": str(log_path), "argv": argv}
    where = f"  (CUDA_VISIBLE_DEVICES={device})" if device is not None else ""
    print("launched:", " ".join(argv) + where)


def log_text(port):
    return pathlib.Path(SERVERS[port]["log_path"]).read_text(errors="replace")


def http_get(url, timeout=10):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, ""
    except Exception as e:
        return 0, str(e)


def http_post(url, payload, timeout=300):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")
    except Exception as e:
        return 0, str(e)


def wait_ready(port, timeout=STARTUP_TIMEOUT_S, every=30):
    """Ready means OUR process is alive AND the port answers — checked in that order.

    ⚠️ The order matters. Checking /health first means ANY server on the port counts as ours. In a
    notebook that is a real hazard: re-run a cell without the cleanup cell, the old server keeps
    answering on 8000, the new one dies with "address already in use", and the run carries on
    against the OLD server with the OLD flags — silently. Found by the dry-run, not in production.
    """
    t0, last = time.time(), 0.0
    while time.time() - t0 < timeout:
        if SERVERS[port]["proc"].poll() is not None:
            rc = SERVERS[port]["proc"].returncode
            print(f"  !! server on {port} EXITED (returncode {rc}). Log tail:")
            print(log_text(port)[-2000:])
            if "address already in use" in log_text(port).lower():
                print("  -> another server already owns this port. Run the clean-up cell, or")
                print("     !pkill -f 'vllm serve'   and start again.")
            return False
        if http_get(f"http://127.0.0.1:{port}/health", 5)[0] == 200:
            print(f"  port {port} ready in {time.time() - t0:.0f}s")
            return True
        waited = time.time() - t0
        if waited - last >= every:
            last = waited
            tail = log_text(port).strip().splitlines()
            print(f"    ...{waited:.0f}s  {tail[-1][:110] if tail else '(log empty)'}")
        time.sleep(5)
    print(f"  !! not ready within {timeout}s")
    return False


def stop_server(port):
    s = SERVERS.pop(port, None)
    if not s:
        return
    s["proc"].terminate()
    try:
        s["proc"].wait(timeout=90)
    except subprocess.TimeoutExpired:
        s["proc"].kill()
    s["logf"].close()


def start_and_wait(port, tag, **kw):
    """Start; if the metrics flags are rejected, restart without them and record that."""
    global METRICS_FLAG_OK
    start_server(port, tag, **kw)
    if wait_ready(port):
        if kw.get("prefix_caching", True) and kw.get("kv_metrics", True):
            METRICS_FLAG_OK = "kv_cache_metrics=True" in log_text(port)
        return True
    if "unrecognized arguments" in log_text(port) and kw.get("kv_metrics", True):
        print("  -> metrics flags rejected by this build; retrying without them")
        METRICS_FLAG_OK = False
        stop_server(port)
        start_server(port, tag + "_nometrics", **{**kw, "kv_metrics": False})
        return wait_ready(port)
    return False


def scrape(port):
    return http_get(f"http://127.0.0.1:{port}/metrics", 30)[1]


def metric(text, family, **labels):
    """Sum a metric family over matching labels. None = ABSENT, which is not the same as 0.0.

    prometheus_client appends _total to counters; hard-coding either spelling is a false negative.
    """
    lab = "".join(f'(?=[^}}]*{k}="{v}")' for k, v in labels.items())
    pat = re.compile(r"^" + re.escape(family) + r"(?:_total)?\{" + lab + r"[^}]*\}\s+([0-9eE+.\-]+)",
                     re.MULTILINE)
    vals = [float(x) for x in pat.findall(text)]
    return sum(vals) if vals else None


def present(text, family):
    return bool(re.search(r"^" + re.escape(family), text, re.MULTILINE))


def complete(port, ids, max_tokens=8):
    return http_post(f"http://127.0.0.1:{port}/v1/completions",
                     {"model": MODEL, "prompt": ids, "max_tokens": max_tokens, "temperature": 0.0})


print("helpers ready")


## 5 — Caching ON: the counters, the unit, the partial-block rule

Re-confirms v1 (it should reproduce **33,344 / 32,768 / 576** exactly) and now also enables the
`kv_block_*` histograms.


In [ ]:
rng = random.Random(0)
prefix = [rng.randrange(1000, 50000) for _ in range(SHARED_PREFIX_TOKENS)]

assert start_and_wait(PORT_A, "caching_on", device=0 if PLANNED_PATH == "multi-gpu" else None)

before = scrape(PORT_A)
q0 = metric(before, "vllm:prefix_cache_queries")
complete(PORT_A, prefix, max_tokens=4)                 # the cold probe
after_probe = scrape(PORT_A)
probe_dq = metric(after_probe, "vllm:prefix_cache_queries") - q0
QUERY_UNIT = ("tokens" if abs(probe_dq - SHARED_PREFIX_TOKENS) <= BLOCK_SIZE
              else "blocks" if abs(probe_dq - SHARED_PREFIX_TOKENS // BLOCK_SIZE) <= 2
              else f"unclear ({probe_dq})")
print(f"probe: {SHARED_PREFIX_TOKENS} tokens sent, queries +{probe_dq}  ->  counts {QUERY_UNIT.upper()}")

t0 = time.time()
for i in range(N_SHARED_REQUESTS):
    complete(PORT_A, prefix + [60000 + i])            # identical prefix + 1 unique token
ON_WALL_S = time.time() - t0
ON = scrape(PORT_A)
pathlib.Path(OUT_DIR, "metrics_caching_on.txt").write_text(ON)

ON_Q = metric(ON, "vllm:prefix_cache_queries")
ON_H = metric(ON, "vllm:prefix_cache_hits")
ON_COMPUTED = metric(ON, "vllm:prompt_tokens_by_source", source="local_compute")
print(f"\nqueries {ON_Q:.0f}   hits {ON_H:.0f}   computed {ON_COMPUTED:.0f}")
print("v1 measured 33344 / 32768 / 576 on the identical workload:",
      "REPRODUCED" if (ON_Q, ON_H, ON_COMPUTED) == (33344, 32768, 576) else "DIFFERENT - investigate")

HIST = {f: present(ON, f) for f in ["vllm:kv_block_lifetime_seconds",
                                     "vllm:kv_block_idle_before_evict_seconds",
                                     "vllm:kv_block_reuse_gap_seconds"]}
print(f"\nengine config says kv_cache_metrics=True: {METRICS_FLAG_OK}")
for k, v in HIST.items():
    print(f"  {'OK' if v else '--'}  {k}")
print("==> H4 is TESTABLE" if all(HIST.values()) else "==> H4 still untestable - see the log")
stop_server(PORT_A)


## 6 — Caching OFF: compare prefill WORK, not hit rate

With caching off vLLM never queries the cache, so its hit rate is undefined — v1 compared an undefined
number and reported *"on does not beat off"*, which was a notebook bug. The honest comparison is
**prefill tokens actually computed** for the same 64 requests.


In [ ]:
time.sleep(5)
assert start_and_wait(PORT_A, "caching_off", prefix_caching=False,
                      device=0 if PLANNED_PATH == "multi-gpu" else None)
b = scrape(PORT_A)
c0 = metric(b, "vllm:prompt_tokens_by_source", source="local_compute") or 0.0
t0 = time.time()
for i in range(N_SHARED_REQUESTS):
    complete(PORT_A, prefix + [60000 + i])
OFF_WALL_S = time.time() - t0
OFF = scrape(PORT_A)
pathlib.Path(OUT_DIR, "metrics_caching_off.txt").write_text(OFF)
OFF_COMPUTED = (metric(OFF, "vllm:prompt_tokens_by_source", source="local_compute") or 0.0) - c0
stop_server(PORT_A)

ON_WORK = N_SHARED_REQUESTS * 1          # prefix already warm: only each unique tail token
print(f"prefill tokens computed for the same {N_SHARED_REQUESTS} requests:")
print(f"  caching ON : {ON_WORK:>7.0f}")
print(f"  caching OFF: {OFF_COMPUTED:>7.0f}")
PREFILL_REDUCTION = OFF_COMPUTED / ON_WORK if ON_WORK else None
print(f"==> {PREFILL_REDUCTION:.0f}x less prefill with caching on")
print(f"\nwall clock, 64 sequential requests: ON {ON_WALL_S:.2f}s   OFF {OFF_WALL_S:.2f}s")
print("  (T4 numbers - TRITON_ATTN + fp16. Direction is meaningful; magnitude does not transfer.)")


## 7 — Two replicas, and the FIRST multi-replica sim-to-real check

On a multi-GPU box each server gets its own physical GPU — the cleanest isolation available, better
than MIG. Then the question this whole project is about: send a **shared-prefix** workload
**round-robin** across both replicas.

### ⭐ Pre-registered prediction — committed 2026-09-22, BEFORE this cell ever ran

`kv-routing-sim`'s own simulator, replaying this exact workload (64 requests, one 512-token prefix,
one unique token each, alternating across 2 replicas), predicts:

| | predicted |
|---|---|
| prefill tokens computed | **1,088** (each replica pays one cold prefix: 2 x 512 + 64 tails) |
| hit rate | **0.966862** |

For contrast, the same workload on ONE replica — or with prefix-affinity routing, which sends it all
to one — computes **576**. So routing without affinity costs **1.89x more prefill** here. That ratio is
the phenomenon `kv-routing-sim` exists to measure, and this cell checks whether the real engine agrees.


In [ ]:
time.sleep(5)
if PLANNED_PATH == "multi-gpu":
    ok_a = start_and_wait(PORT_A, "replica_a", device=0)
    ok_b = start_and_wait(PORT_B, "replica_b", device=1)
elif PLANNED_PATH == "mig":
    ok_a = start_and_wait(PORT_A, "replica_a", device=MIG_UUIDS[0])
    ok_b = start_and_wait(PORT_B, "replica_b", device=MIG_UUIDS[1])
else:
    ok_a = start_and_wait(PORT_A, "replica_a", mem=GPU_MEM_UTIL_SPLIT)
    ok_b = start_and_wait(PORT_B, "replica_b", mem=GPU_MEM_UTIL_SPLIT)
a_still = http_get(f"http://127.0.0.1:{PORT_A}/health", 10)[0] == 200 if ok_a else False
COEXIST = bool(ok_a and ok_b and a_still)
print(f"\npath {PLANNED_PATH}: A {ok_a}  B {ok_b}  A still healthy {a_still}  ->  coexist {COEXIST}")

RR = None
if COEXIST:
    # Round-robin a fresh shared-prefix workload across both replicas.
    rr_prefix = [rng.randrange(1000, 50000) for _ in range(SHARED_PREFIX_TOKENS)]
    s_a0, s_b0 = scrape(PORT_A), scrape(PORT_B)
    for i in range(N_SHARED_REQUESTS):
        complete(PORT_A if i % 2 == 0 else PORT_B, rr_prefix + [70000 + i])
    s_a1, s_b1 = scrape(PORT_A), scrape(PORT_B)
    pathlib.Path(OUT_DIR, "metrics_replica_a.txt").write_text(s_a1)
    pathlib.Path(OUT_DIR, "metrics_replica_b.txt").write_text(s_b1)

    def d(new, old, fam, **lab):
        return (metric(new, fam, **lab) or 0.0) - (metric(old, fam, **lab) or 0.0)

    q = d(s_a1, s_a0, "vllm:prefix_cache_queries") + d(s_b1, s_b0, "vllm:prefix_cache_queries")
    h = d(s_a1, s_a0, "vllm:prefix_cache_hits") + d(s_b1, s_b0, "vllm:prefix_cache_hits")
    comp = (d(s_a1, s_a0, "vllm:prompt_tokens_by_source", source="local_compute")
            + d(s_b1, s_b0, "vllm:prompt_tokens_by_source", source="local_compute"))
    RR = {"queries": q, "hits": h, "computed": comp, "hit_rate": h / q if q else None}
    print(f"\nround-robin across 2 replicas: queries {q:.0f}  hits {h:.0f}  computed {comp:.0f}")
    print(f"  hit rate {RR['hit_rate']:.4f}")
    print(f"  expected: each replica pays ONE cold prefix -> computed = 2 x {SHARED_PREFIX_TOKENS} "
          f"+ {N_SHARED_REQUESTS} tails = {2 * SHARED_PREFIX_TOKENS + N_SHARED_REQUESTS}")
    PREDICTED = {"computed": 1088, "hit_rate": 0.966862}   # from kv-routing-sim, pre-registered
    RR["predicted"] = PREDICTED
    RR["computed_error"] = comp - PREDICTED["computed"]
    RR["hit_rate_error"] = (RR["hit_rate"] - PREDICTED["hit_rate"]) if RR["hit_rate"] else None
    print(f"\n  SIM-TO-REAL   predicted computed {PREDICTED['computed']}   measured {comp:.0f}   "
          f"error {RR['computed_error']:+.0f} tokens")
    print(f"                predicted hit rate {PREDICTED['hit_rate']:.6f}   measured {RR['hit_rate']:.6f}")
    if RR["computed_error"] == 0:
        print("  ==> EXACT. The simulator predicted the real engine's 2-replica prefill to the token.")
    else:
        print("  ==> MISMATCH. Do not explain it away - record it. Likely suspects: request")
        print("      interleaving (vLLM may batch requests before the prefix is cached), or")
        print("      chunked prefill splitting the first request. Both are real H3/H4 material.")
    print("\n  (one replica, or prefix-affinity routing, computes 576 on this workload - so")
    print("   round-robin costs 1.89x more prefill. That ratio is what kv-routing-sim measures.)")


## 8 — Save the verdict


In [ ]:
GATE = {
    "vllm_version": VLLM_VERSION,
    "query_unit": QUERY_UNIT,
    "reproduced_v1_counters": (ON_Q, ON_H, ON_COMPUTED) == (33344, 32768, 576),
    "prefill_reduction_x": PREFILL_REDUCTION,
    "kv_cache_metrics_enabled": METRICS_FLAG_OK,
    "kv_block_histograms_present": HIST,
    "H4_testable": all(HIST.values()),
    "two_replica_path": PLANNED_PATH,
    "two_replicas_coexist": COEXIST,
    "round_robin_two_replicas": RR,
    "timing_hardware_caveat": ("T4/SM75: TRITON_ATTN + fp16. Cache accounting is valid; TTFT and "
                               "cost coefficients are T4-only and do not transfer to A100/H100."),
}
payload = {"script": "notebooks/phase0_spike_vllm.ipynb (v2)", "model": MODEL,
           "block_size": BLOCK_SIZE, "max_model_len": MAX_MODEL_LEN, "environment": ENV,
           "metric_name_audit": AUDIT, "gate": GATE,
           "on": {"queries": ON_Q, "hits": ON_H, "computed": ON_COMPUTED, "wall_s": ON_WALL_S},
           "off": {"computed": OFF_COMPUTED, "wall_s": OFF_WALL_S}}
pathlib.Path(OUT_DIR, "phase0_v2.json").write_text(json.dumps(payload, indent=2, default=str))
print(json.dumps(GATE, indent=2, default=str))
print("\nfiles to copy into artifacts/results/:")
for f in sorted(pathlib.Path(OUT_DIR).iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} B)")


## 9 — Clean up

⚠️ **Run this.** A forgotten `vllm serve` bills until the box dies.


In [ ]:
for port in list(SERVERS):
    stop_server(port)
print(sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv"))
